# 070 CASE 040 — Grazing classification: Munkarp (Höör, Skåne)

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TobiasEdman/imintengine/main?labpath=notebooks%2Fsdl3-cases%2F070_CASE_040-Grazing-Munkarp.ipynb)

**Purpose:** Automate EU agricultural-support (CAP) compliance checks — classify declared grazing parcels (LPIS blocks) as actively grazed or not, using a Sentinel-2 timeseries through the growing season.

**Partners:**
- **Jordbruksverket (SJV)** — LPIS (Land Parcel Identification System, open data via WFS)
- **Naturvårdsverket** — NMD reference
- **RISE** — `pib-ml-grazing` CNN-biLSTM model

**Licence:** CC0 1.0 notebook. LPIS data CC BY 4.0 (Jordbruksverket). Model MIT (RISE, 2025).

## 1. Method

```
AOI bbox  →  fetch_lpis_polygons (Jordbruksverket WFS, open)
          →  fetch_grazing_timeseries (DES/CDSE Sentinel-2, 12-band × T dates)
          →  GrazingAnalyzer.predict_batch (CNN-biLSTM)
          →  per-polygon class + confidence
```

`GrazingAnalyzer` is a standalone class (not in `ANALYZER_REGISTRY`) — it consumes `GrazingTimeseriesResult` objects rather than `IMINTJob`.

## 2. Setup and LPIS fetch

In [ ]:
import matplotlib.pyplot as plt
import folium
from imint.fetch import fetch_lpis_polygons
from imint.analyzers.grazing import GrazingAnalyzer

# Munkarp (Höör, Skåne) — small AOI with ~60 pasture blocks
AOI = {"west": 13.42, "south": 55.935, "east": 13.48, "north": 55.965}
YEAR = 2025

print(f"AOI (Munkarp): {AOI}")
print(f"Reference year: {YEAR}")

# Real LPIS pastures — no authentication required
lpis = fetch_lpis_polygons(AOI, agoslag="Bete", max_features=500)
print(f"\nLPIS pasture polygons fetched: {len(lpis)}")
print(f"CRS: {lpis.crs}")
if len(lpis):
    print(f"Total pasture area: {lpis.geometry.area.sum() / 1e6:.2f} km²")
    print(f"Columns: {list(lpis.columns)}")

## 3. Run the grazing classifier

Full pipeline calls `fetch_grazing_timeseries` (needs DES/CDSE, ~5 min for 60 polygons). For an always-reproducible smoke-test we fall back to a synthetic timeseries per polygon when the fetch fails.

In [ ]:
import numpy as np

try:
    from imint.fetch import fetch_grazing_timeseries
    # Live path — limit to first 5 polygons so notebook runs in < 2 min
    subset = lpis.head(5).to_crs("EPSG:4326")
    ts_list = fetch_grazing_timeseries(
        polygons=subset,
        year=YEAR,
        polygon_id_col="blockid" if "blockid" in subset.columns else None,
        source="des",
    )
    print(f"Fetched {len(ts_list)} live timeseries")
except Exception as exc:
    print(f"Live fetch unavailable ({type(exc).__name__}: {exc})")
    print("Falling back to synthetic timeseries for model smoke-test.")

    class _FakeTS:
        def __init__(self, polygon_id, data, dates):
            self.polygon_id = polygon_id
            self.data = data
            self.dates = dates

    rng = np.random.default_rng(42)
    ts_list = [
        _FakeTS(
            polygon_id=f"munkarp-synth-{i}",
            data=rng.normal(0.3, 0.1, size=(19, 12, 46, 46)).astype(np.float32),
            dates=[f"2025-{m:02d}-15" for m in range(3, 11)],
        )
        for i in range(5)
    ]

analyzer = GrazingAnalyzer()
predictions = analyzer.predict_batch(ts_list)

active = sum(1 for p in predictions if p.predicted_class == 1)
inactive = sum(1 for p in predictions if p.predicted_class == 0)
errors = sum(1 for p in predictions if p.predicted_class == -1)
print(f"\nActive grazing: {active}")
print(f"No activity:    {inactive}")
print(f"Errors:         {errors}")
if predictions and errors < len(predictions):
    mean_conf = np.mean([p.confidence for p in predictions if p.predicted_class >= 0])
    print(f"Mean confidence: {mean_conf:.3f}")

## 4. Visualize — LPIS polygons coloured by prediction

In [ ]:
center = [(AOI["south"] + AOI["north"]) / 2, (AOI["west"] + AOI["east"]) / 2]
m = folium.Map(location=center, zoom_start=13, tiles="OpenStreetMap")
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Satellite",
).add_to(m)

color_map = {1: "#2ca02c", 0: "#d62728", -1: "#ff7f0e"}
# Match predictions back onto the LPIS GeoDataFrame
id_to_cls = {p.polygon_id: p.predicted_class for p in predictions}
for _, row in lpis.to_crs("EPSG:4326").iterrows():
    pid = row.get("blockid", None)
    cls = id_to_cls.get(pid, None)
    color = color_map.get(cls, "#888")
    folium.GeoJson(
        row.geometry.__geo_interface__,
        style_function=lambda _, c=color: {
            "fillColor": c, "color": c, "weight": 1.5, "fillOpacity": 0.35,
        },
        popup=f"block_id={pid} cls={cls}",
    ).add_to(m)

folium.LayerControl().add_to(m)
m

In [ ]:
# Static bar chart — prediction summary
fig, ax = plt.subplots(figsize=(6, 4))
labels = ["Active grazing", "No activity", "Error"]
counts = [active, inactive, errors]
colors = ["#2ca02c", "#d62728", "#ff7f0e"]
ax.bar(labels, counts, color=colors)
ax.set_ylabel("Polygon count")
ax.set_title(f"Munkarp grazing classification — {YEAR}")
for i, c in enumerate(counts):
    ax.text(i, c + 0.1, str(c), ha="center")
plt.tight_layout()
plt.show()

## 5. Interpretation

**Operational value:**
Jordbruksverket is obliged by EU rules to check declared area-aid. Field inspection covers ≤2% of blocks; satellite classification covers 100% so human inspections can focus on anomalies.

**Caveats:**
- Model is **decision support**, not automated decision-making.
- ~5% classification error → stratified field sampling still needed for calibration.
- Early mowing, drought or flooding can produce false "no activity" readings.

**Scaling up:**
- Remove the `.head(5)` cap to run on the full AOI (~60 polygons, ~15 min)
- Increase `YEAR` range for multi-year trend analysis
- Switch `source="cdse_http"` for faster fetch on larger AOIs

### Links
- [`imint/fetch.py::fetch_lpis_polygons`](https://github.com/TobiasEdman/imintengine/blob/main/imint/fetch.py)
- [`imint/fetch.py::fetch_grazing_timeseries`](https://github.com/TobiasEdman/imintengine/blob/main/imint/fetch.py)
- [`imint/analyzers/grazing.py`](https://github.com/TobiasEdman/imintengine/blob/main/imint/analyzers/grazing.py)
- [LPIS open data — Jordbruksverket](https://jordbruksverket.se)